## Code: Data Splitter.
## Dissertation: Real-Time Federated Learning Framework for Detecting Cardiovascular diseases in ECG signals.
## Made by: Juan David Velasquez R.

**Description**: The aim of this code is to split the data into three parts, which correspond to the three clients or models of the Federated Learning Framework. Additionally, the remaining data will be used as the test set, also known as "Test Split". Likewise, the output will be 4 folders, 3 correspond to the npz files of each client and the fourth one, the test set.

The first section below is associated to the libraries import, required for ensuring a proper functioning of the code.

In [ ]:
import numpy as np # Used for array operations.
import os # Used for file operations.
from pathlib import Path # Used for path operations.
import random # Used for randomization.
import shutil # Used for file operations.

The following portion of code corresponds to the function that generates the splits, which is crucial for ensuring that each client holds the same amount of data. Additionally, generates the test set for evaluating the Federated framework implemented.

In [ ]:
def create_splits(data, labels, output_dir="Split_Data", seed=42):
    """
    Shuffles the dataset, splits it into three 30% training parts and one 10% test set,
    and saves each sample individually as a .npz file named 'ecg_ID.npz' in separate folders.

    Args:
        data: Numpy array of shape (n_samples, 8, 600).
        labels: Numpy array of shape (n_samples).
        output_dir: Base directory to save split folders.
        seed: Random seed for reproducibility.
    """
    assert data.shape[0] == labels.shape[0], "Data and labels must have the same number of samples."

    # Shuffle data
    np.random.seed(seed)
    indices = np.random.permutation(data.shape[0]) # Shuffle indices of the input data.
    data = data[indices] # Shuffle data.
    labels = labels[indices] # Shuffle labels.

    # Define split sizes.
    total_samples = data.shape[0]
    size_30 = int(0.30 * total_samples)

    # Define partition ranges.
    partitions = {
        "model1": (0, size_30), # 30% for first client.
        "model2": (size_30, 2 * size_30), # 30% for second client.
        "model3": (2 * size_30, 3 * size_30), # 30% for third client.
        "test": (3 * size_30, total_samples) # Remaining 10% for clients testing.
    }

    # Save each sample individually as ecg_<global_id>.npz.
    global_id = 0
    for folder_name, (start, end) in partitions.items(): # This for iterates over the partitions dictionary, considering the folder name and the start and end indices of the samples that will be saved in the folder.
        folder_path = os.path.join(output_dir, folder_name) # This variable contains the path to the folder that will contain the output files of the model. In this case, will be 
        # "Split_Data/model1", "Split_Data/model2", "Split_Data/model3" and "Split_Data/test".
        os.makedirs(folder_path, exist_ok=True) # Creates the folder if it doesn't exist.

        for idx in range(start, end): # This for iterates over the samples (data array) that will be saved in the folder, considering the start and end indices, given by the partitions dictionary.
            filename = os.path.join(folder_path, f"ecg_{global_id}.npz") # This variable holds the path and filename of the file to be generated.
            np.savez(filename, x=data[idx], y=labels[idx]) # Saves the file.
            global_id += 1 # Increments the global ID.

        print(f"Saved {end - start} samples to {folder_path}") # Confirmation message.


### Performing the Data Split

In [ ]:
data, labels = [], [] # Creates two empty lists that will contain the full dataset and the labels.

for files in os.listdir('Data_Preproc'): # This for iterates over the folder that contains the npz files.
    if files.endswith('.npz'): # Checks if the file is a npz file.
        data.append((np.load(os.path.join('Data_Preproc', files))['x'].copy()).transpose(0, 2, 1)) # Here the data is read and appended to the data list. Additionally, it is transposed because the LSTM model needs to 
        # capture the data in the format of n_samples, 600, 8. 
        labels.append(np.load(os.path.join('Data_Preproc', files))['y']) # The labels are read and appended to the labels list.
    else:
        continue

create_splits(data, labels) # Calls the create_splits function and generates the four folders of data.

Saved 25760 samples to Split_Data\model1
Saved 25760 samples to Split_Data\model2
Saved 25760 samples to Split_Data\model3
Saved 8587 samples to Split_Data\test
